#  Clipe Musical com IA — Fase 2
## Geração de Frames com Stable Diffusion + IP-Adapter
###  100% Gratuito · Google Colab GPU T4

**Ordem obrigatória das células: 1 → reinicia → 2 → 3 → 4 → 5 → 6 → 7 → 8**

| Tarefa | Ferramenta |
|---|---|
| Geração de imagens | Stable Diffusion 1.5 |
| Consistência do personagem | IP-Adapter Plus Face |
| Detecção facial | InsightFace / OpenCV |

---
 **Obrigatório:** `Ambiente de execução → Alterar tipo → GPU T4`

## 📦 Célula 1 — Instalação
> Após rodar, **reinicie a sessão** antes de continuar.

In [ ]:
print('Instalando dependências...\n')
!pip install -q --upgrade diffusers transformers accelerate safetensors
!pip install -q insightface onnxruntime-gpu opencv-python-headless Pillow numpy
print('\n Instalação concluída!')
print('  Agora vá em: Ambiente de execução → Reiniciar sessão')
print('   Depois continue a partir da Célula 2.')


##  Célula 2 — Upload do ZIP da Fase 1 + Imagens

In [ ]:
from google.colab import files
from IPython.display import display, Image as IPImage
import os, shutil, zipfile, json

BASE_DIR   = '/content/fase2'
INPUT_DIR  = f'{BASE_DIR}/input'
IMG_DIR    = f'{BASE_DIR}/imagens'
OUTPUT_DIR = f'{BASE_DIR}/frames'
FACE_DIR   = f'{BASE_DIR}/faces'

for d in [INPUT_DIR, IMG_DIR, OUTPUT_DIR, FACE_DIR]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

# ── ZIP da Fase 1 ─────────────────────────────────────────────
print('═' * 55)
print('  PASSO 1 — Upload do ZIP gerado pela Fase 1')
print('═' * 55)
uploaded_zip = files.upload()

ZIP_PATH = None
for filename, content in uploaded_zip.items():
    if not filename.lower().endswith('.zip'):
        print(f'  "{filename}" ignorado — apenas .zip')
        continue
    dest = os.path.join(INPUT_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(content)
    ZIP_PATH = dest
    print(f' ZIP: {filename}  ({len(content)/(1024*1024):.1f} MB)')

if ZIP_PATH is None:
    raise ValueError('❌ Nenhum .zip enviado.')

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(INPUT_DIR)
print('\n Conteúdo extraído:')
for fn in sorted(os.listdir(INPUT_DIR)):
    print(f'   {fn}')

# Carrega JSONs automaticamente pelo sufixo
storyboard      = None
perfil_visual   = None
analise_musical = None
nome_musica     = None

for fn in os.listdir(INPUT_DIR):
    path = os.path.join(INPUT_DIR, fn)
    if not fn.endswith('.json'): continue
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    if fn.endswith('_storyboard.json'):
        storyboard  = data
        nome_musica = fn.replace('_storyboard.json', '')
    elif fn.endswith('_perfil_visual.json'):
        perfil_visual = data
    elif fn.endswith('_analise.json'):
        analise_musical = data

if storyboard is None:
    raise ValueError('❌ storyboard JSON não encontrado no ZIP.')

print(f'\n Projeto: "{nome_musica}"')
print(f'   Cenas  : {len(storyboard["cenas"])}')
print(f'   Conceito: {storyboard["conceito_geral"][:80]}')

# ── Imagens do personagem ──────────────────────────────────────
print()
print('═' * 55)
print('  PASSO 2 — Upload das imagens do personagem (mín. 4)')
print('  Formatos: .jpg  .jpeg  .png')
print('═' * 55)
uploaded_imgs = files.upload()

IMAGE_PATHS = []
for filename, content in uploaded_imgs.items():
    ext = os.path.splitext(filename)[1].lower()
    if ext not in {'.jpg', '.jpeg', '.png'}:
        print(f'  "{filename}" ignorado.')
        continue
    dest = os.path.join(IMG_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(content)
    IMAGE_PATHS.append(dest)
    print(f'  ✓ {filename}  ({len(content)/1024:.0f} KB)')

if len(IMAGE_PATHS) < 4:
    raise ValueError(f'❌ Mínimo 4 imagens. Você enviou {len(IMAGE_PATHS)}.')

print(f'\n {len(IMAGE_PATHS)} imagens prontas!')
for p in IMAGE_PATHS:
    display(IPImage(filename=p, width=180))


##  Célula 3 — Extração do melhor rosto de referência

In [ ]:
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, Image as IPImage
import os

print(' Detectando rostos nas imagens...\n')

FACE_REF_PATH = None
candidates    = []

try:
    from insightface.app import FaceAnalysis
    app = FaceAnalysis(name='buffalo_l',
                       providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640, 640))

    for img_path in IMAGE_PATHS:
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: continue
        for face in app.get(img_bgr):
            score = float(face.det_score)
            bbox  = face.bbox.astype(int)
            area  = int((bbox[2]-bbox[0]) * (bbox[3]-bbox[1]))
            candidates.append({'path': img_path, 'score': score * np.sqrt(area),
                                'det_score': score, 'bbox': bbox, 'area': area})
            print(f'  ✓ {os.path.basename(img_path):30s}  score={score:.3f}  área={area}px²')

    metodo = 'InsightFace'
except Exception as e:
    print(f'  InsightFace indisponível ({e}), usando OpenCV...')
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    for img_path in IMAGE_PATHS:
        img  = cv2.imread(img_path)
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        for (x, y, fw, fh) in cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60)):
            area = int(fw * fh)
            candidates.append({'path': img_path, 'score': area,
                                'det_score': 1.0, 'bbox': [x, y, x+fw, y+fh], 'area': area})
            print(f'  ✓ {os.path.basename(img_path):30s}  área={area}px²')
    metodo = 'OpenCV'

if candidates:
    best    = max(candidates, key=lambda x: x['score'])
    img_bgr = cv2.imread(best['path'])
    x1, y1, x2, y2 = best['bbox']
    h, w = img_bgr.shape[:2]
    pad  = int(max(x2-x1, y2-y1) * 0.4)
    crop = img_bgr[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)]
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    face_img = Image.fromarray(crop_rgb).resize((512, 512))
    FACE_REF_PATH = os.path.join(FACE_DIR, 'face_reference.png')
    face_img.save(FACE_REF_PATH)
    print(f'\n✅ [{metodo}] Melhor rosto: {os.path.basename(best["path"])}  score={best["det_score"]:.3f}')
else:
    print(' Nenhum rosto detectado. Usando primeira imagem como referência.')
    img = Image.open(IMAGE_PATHS[0]).convert('RGB').resize((512, 512))
    FACE_REF_PATH = os.path.join(FACE_DIR, 'face_reference.png')
    img.save(FACE_REF_PATH)

print('\n  Imagem de referência (512×512):')
display(IPImage(filename=FACE_REF_PATH, width=256))


##  Célula 4 — Configurações de geração
> Ajuste aqui antes de carregar os modelos.

In [ ]:
# ── Configurações — edite à vontade ───────────────────────────
WIDTH            = 768   # largura dos frames
HEIGHT           = 512   # altura dos frames
FRAMES_POR_CENA  = 5     # frames por cena
NUM_STEPS        = 45    # aumentado: menos artefatos em 768x512
GUIDANCE_SCALE   = 6.0   # reduzido: menos distorção anatômica
IPADAPTER_SCALE  = 0.5   # rosto mais natural, menos colado
SEED_BASE        = 42

NEGATIVE_GLOBAL = (
    'blurry, low quality, distorted, ugly, bad anatomy, bad proportions, '
    'deformed, watermark, text, extra limbs, duplicate, disfigured, '
    'dark eyes, black eyes, asymmetric eyes, cross-eyed, fused fingers, '
    'extra fingers, poorly drawn hands, poorly drawn face, mutation'
)
# ──────────────────────────────────────────────────────────────

total_frames = FRAMES_POR_CENA * len(storyboard['cenas'])
est_min      = total_frames * 20 / 60  # ~20s/frame com 45 steps

print('  Configurações:')
print(f'   Resolução       : {WIDTH}×{HEIGHT}px')
print(f'   Frames/cena     : {FRAMES_POR_CENA}')
print(f'   Total de frames : {total_frames}')
print(f'   Steps           : {NUM_STEPS}  (↑ menos artefatos)')
print(f'   Guidance        : {GUIDANCE_SCALE}  (↓ menos distorção)')
print(f'   IP-Adapter scale: {IPADAPTER_SCALE}')
print(f'\n Tempo estimado na T4: ~{est_min:.0f}–{est_min*1.3:.0f} min')


##  Célula 5 — Carrega SD 1.5 + IP-Adapter
>  *Download ~4 GB na primeira vez. Nas próximas usa o cache.*

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler
from transformers import CLIPTokenizer
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  {device.upper()} — {torch.cuda.get_device_name(0) if device=="cuda" else "CPU"}')

# ── Realistic Vision V6 — muito melhor para pessoas realistas ─
# Substitui o SD 1.5 base que gera anatomia ruim
print('\n⬇  Carregando Realistic Vision V6...')
pipe = StableDiffusionPipeline.from_pretrained(
    'SG161222/Realistic_Vision_V6.0_B1_noVAE',
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
).to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
print(' Realistic Vision V6 pronto!')

# ── IP-Adapter Full Face ───────────────────────────────────────
print('\n🔧 Carregando IP-Adapter Full Face...')
pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='models',
    weight_name='ip-adapter-full-face_sd15.bin'
)
pipe.set_ip_adapter_scale(IPADAPTER_SCALE)
print(f' IP-Adapter pronto!  scale={IPADAPTER_SCALE}')

# ── Truncagem de prompt ────────────────────────────────────────
tokenizer_clip = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')

def truncar_prompt(texto, max_tokens=75):
    ids = tokenizer_clip.encode(texto, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return texto
    return tokenizer_clip.decode(ids[:max_tokens], skip_special_tokens=True)

# ── Imagem de referência ──────────────────────────────────────
REF_IMAGE = Image.open(FACE_REF_PATH).convert('RGB')
print(f'\n Referência: {FACE_REF_PATH}  tamanho={REF_IMAGE.size}')


##  Célula 6 — Geração dos frames
>  *Célula mais demorada. Acompanhe o progresso pelo log.*

In [ ]:
import torch, json, time, os
from PIL import Image
from IPython.display import display, Image as IPImage

log_geracao = {
    'nome_musica': nome_musica,
    'config': {
        'width': WIDTH, 'height': HEIGHT,
        'frames_por_cena': FRAMES_POR_CENA,
        'steps': NUM_STEPS, 'guidance': GUIDANCE_SCALE,
        'ipadapter_scale': IPADAPTER_SCALE, 'seed_base': SEED_BASE
    },
    'cenas': []
}

total_cenas  = len(storyboard['cenas'])
total_frames = total_cenas * FRAMES_POR_CENA
frame_global = 0
t_inicio     = time.time()

print(f' Gerando {total_frames} frames para "{nome_musica}"...')
print(f'   {total_cenas} cenas × {FRAMES_POR_CENA} frames  |  {WIDTH}×{HEIGHT}px')
print('─' * 60)

for cena in storyboard['cenas']:
    cena_id     = cena['secao_id']
    intensidade = cena['intensidade']

    # Trunca prompts para 77 tokens (limite do CLIP)
    prompt_sd  = truncar_prompt(cena['prompt_sd'])
    neg_raw    = cena.get('negative_prompt', '') + ', ' + NEGATIVE_GLOBAL
    neg_prompt = truncar_prompt(neg_raw)

    # Ajusta parâmetros pela intensidade
    guidance_cena  = GUIDANCE_SCALE + (1.0 if intensidade == 'alta' else 0.0)
    ipa_scale_cena = max(0.3, IPADAPTER_SCALE - (0.1 if intensidade == 'alta' else 0.0))
    pipe.set_ip_adapter_scale(ipa_scale_cena)

    cena_dir = os.path.join(OUTPUT_DIR, f'cena_{cena_id:02d}')
    os.makedirs(cena_dir, exist_ok=True)
    frames_salvos = []

    print(f'\n Cena {cena_id}/{total_cenas}  [{cena["inicio"]}s→{cena["fim"]}s]  {intensidade.upper()}')
    print(f'   Mood  : {cena["mood"]}')
    print(f'   Plano : {cena["tipo_de_plano"]}')
    print(f'   IPA   : {ipa_scale_cena:.2f}  Guidance: {guidance_cena}')
    print(f'   Tokens prompt: {len(tokenizer_clip.encode(prompt_sd))} | neg: {len(tokenizer_clip.encode(neg_prompt))}')

    for fi in range(FRAMES_POR_CENA):
        seed      = SEED_BASE + (cena_id * 100) + fi
        generator = torch.Generator(device=device).manual_seed(seed)
        t_frame   = time.time()

        result = pipe(
            prompt=prompt_sd,
            negative_prompt=neg_prompt,
            ip_adapter_image=REF_IMAGE,
            num_inference_steps=NUM_STEPS,
            guidance_scale=guidance_cena,
            width=WIDTH,
            height=HEIGHT,
            generator=generator,
            num_images_per_prompt=1
        )

        frame_path = os.path.join(cena_dir, f'frame_{fi+1:02d}_seed{seed}.png')
        result.images[0].save(frame_path)
        frames_salvos.append(frame_path)

        frame_global += 1
        elapsed  = time.time() - t_inicio
        restante = (elapsed / frame_global) * (total_frames - frame_global)
        print(f'   [{frame_global:3d}/{total_frames}] frame {fi+1}  '
              f'seed={seed}  {time.time()-t_frame:.1f}s  '
              f'| restante ~{restante/60:.1f} min')

        if fi == 0:
            display(IPImage(filename=frame_path, width=384))

    log_geracao['cenas'].append({
        'secao_id':      cena_id,
        'inicio':        cena['inicio'],
        'fim':           cena['fim'],
        'intensidade':   intensidade,
        'mood':          cena['mood'],
        'tipo_de_plano': cena['tipo_de_plano'],
        'prompt_sd':     prompt_sd,
        'frames':        frames_salvos
    })

log_path = os.path.join(OUTPUT_DIR, f'{nome_musica}_frames_log.json')
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(log_geracao, f, ensure_ascii=False, indent=2)

total_elapsed = time.time() - t_inicio
print('\n' + '═'*60)
print(f' Geração concluída!')
print(f'   Frames : {frame_global}  |  Tempo: {total_elapsed/60:.1f} min  |  Média: {total_elapsed/frame_global:.1f}s/frame')
print('═'*60)


##  Célula 7 — Preview grid de todos os frames

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import os

rows = len(log_geracao['cenas'])
cols = FRAMES_POR_CENA

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.3))
fig.suptitle(f'Frames — {nome_musica}', fontsize=13, y=1.01)

cor_int = {'baixa': '#90CAF9', 'média': '#FFB74D', 'alta': '#EF5350'}

for i, cena in enumerate(log_geracao['cenas']):
    for j, fp in enumerate(cena['frames']):
        ax = axes[i][j] if rows > 1 else axes[j]
        ax.imshow(Image.open(fp))
        ax.axis('off')
        ax.set_title(f'F{j+1}', fontsize=7, pad=2)
        if j == 0:
            cor = cor_int.get(cena['intensidade'], 'gray')
            ax.set_ylabel(
                f"C{cena['secao_id']} {cena['intensidade']}",
                fontsize=8, color=cor, rotation=0, labelpad=50, va='center'
            )

plt.tight_layout()
grid_path = os.path.join(OUTPUT_DIR, f'{nome_musica}_grid.png')
plt.savefig(grid_path, dpi=110, bbox_inches='tight')
plt.show()
print(f' Grid salvo: {grid_path}')


##  Célula 8 — Download dos frames + log

In [ ]:
import zipfile, os
from google.colab import files

zip_path = f'/content/fase2/{nome_musica}_fase2_frames.zip'
print(f' Empacotando frames de "{nome_musica}"...')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for cena in log_geracao['cenas']:
        for fp in cena['frames']:
            arcname = f"cena_{cena['secao_id']:02d}/{os.path.basename(fp)}"
            z.write(fp, arcname)
        print(f"   ✓ Cena {cena['secao_id']:02d} — {len(cena['frames'])} frames")
    z.write(log_path,   os.path.basename(log_path))
    z.write(grid_path,  os.path.basename(grid_path))
    z.write(FACE_REF_PATH, 'face_reference.png')

size_mb = os.path.getsize(zip_path) / (1024*1024)
print(f'\n Baixando {os.path.basename(zip_path)}  ({size_mb:.1f} MB)...')
files.download(zip_path)

print('\n Fase 2 concluída!')
print('═'*50)
print(f'  Projeto  : {nome_musica}')
print(f'  Frames   : {sum(len(c["frames"]) for c in log_geracao["cenas"])}')
print(f'  Resolução: {WIDTH}×{HEIGHT}px')
print(f'  Cenas    : {len(log_geracao["cenas"])}')
print('═'*50)
print('\n  Próxima etapa: Fase 3 — Lip Sync + Animação + FFmpeg')


---
##  Estrutura do ZIP gerado

```
nomeDaMusica_fase2_frames.zip
├── cena_01/
│   ├── frame_01_seed142.png
│   └── ... (5 frames)
├── cena_02/ ... cena_08/
├── face_reference.png
├── nomeDaMusica_frames_log.json
└── nomeDaMusica_grid.png
```

##  Ajustes rápidos
| Quer mudar | Edite na Célula |
|---|---|
| Força do rosto | `IPADAPTER_SCALE` — Célula 4 |
| Qualidade | `NUM_STEPS` — Célula 4 |
| Quantidade de frames | `FRAMES_POR_CENA` — Célula 4 |